## Prompt Evaluation

**Prompt evaluation** (an *eval*) is the practice of testing a prompt against a set of realistic inputs and scoring the results, so that improving a prompt becomes a measurable process instead of a gut feeling.

Without an eval, a prompt that "looks good" on one or two examples can quietly fail on others. With one, every change you make to a prompt produces a number you can compare against the previous version.

A typical eval workflow has five steps:

1. **Draft a prompt** — the prompt you want to test (for example, "solve this AWS task with Python, JSON or a regex").
2. **Create an eval dataset** — a set of test cases, each one an input the prompt should handle.
3. **Run each test case** — merge every test case into the prompt and send it to Claude.
4. **Grade the outputs** — give every response a score (we will use a model grader and a code grader).
5. **Iterate** — change the prompt, re-run the eval, and compare the average score.

In this notebook we build that whole pipeline, one piece at a time:

1. Generating test datasets
2. Running the eval
3. Model-based grading
4. Code-based grading
5. Putting it all together

In [1]:
%%capture
%pip install anthropic python-dotenv

In [2]:
# Setup: load environment variables and create the client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-5-5"

## Helper Functions

Every section below talks to Claude through two small helpers, so we define them once here:

- **`add_user_message`** appends a user message to the conversation list.
- **`chat`** sends the conversation to Claude and returns the text of the reply. Pass a JSON `schema` and the API guarantees the reply is valid JSON that matches it. We use this to get clean JSON back in sections 1 and 3.

> **Note:** `max_tokens` is `4000` because `claude-haiku-5-5` thinks before it answers, and that thinking counts toward the limit.

In [3]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def chat(messages, system=None, schema=None):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    # With a schema, the reply is guaranteed to be valid JSON that matches it
    if schema:
        params["output_config"] = {"format": {"type": "json_schema", "schema": schema}}

    message = client.messages.create(**params)
    # Join the text blocks: the reply may start with a thinking block
    return "".join(block.text for block in message.content if block.type == "text")

## 1. Generating Test Datasets

**Definition:** An eval dataset is a list of test cases — the inputs we will feed to our prompt. Each test case is a small JSON object, and its fields depend on what we are testing.

Writing a dataset by hand is slow, so we can ask Claude to generate one for us. Our running example is a prompt that writes **Python, JSON, or a regular expression** for AWS-related tasks, so each test case has:

- **`task`** — a description of what needs to be written.
- **`format`** — which of `"python"`, `"json"` or `"regex"` the answer must be (we will use this for grading in section 4).

To get *only* JSON back (no "Here is your dataset:" preamble) we use **structured outputs**:

- We describe the exact shape we want as a **JSON schema** and pass it in `output_config`.
- The API guarantees the reply is valid JSON that matches the schema, so it can go straight into `json.loads`.
- The top level of the schema is an object, so the list of test cases sits inside a `dataset` key.
- `enum` limits `format` to the three allowed values.

> **Note:** Older models were given the start of the reply (a *prefill* such as ``"```json"``) plus a stop sequence. `claude-haiku-5-5` rejects prefill with a 400 error, and structured outputs replace it. See `08-structureddata.ipynb` for both methods side by side.

In [4]:
# Function to generate a new dataset
import json

DATASET_SCHEMA = {
    "type": "object",
    "properties": {
        "dataset": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "task": {"type": "string"},
                    "format": {"type": "string", "enum": ["json", "python", "regex"]},
                },
                "required": ["task", "format"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["dataset"],
    "additionalProperties": False,
}


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of objects,
each representing task that requires Python, JSON, or a Regex to complete.

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    text = chat(messages, schema=DATASET_SCHEMA)
    return json.loads(text)["dataset"]

In [5]:
# Generate the dataset and write it to 'dataset.json'
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

dataset

[{'task': 'Write a single Python function using boto3 that takes a bucket name and returns the number of objects stored in that S3 bucket.',
  'format': 'python'},
 {'task': "Write a JSON object representing an AWS IAM policy that allows read-only access (s3:GetObject and s3:ListBucket) to the S3 bucket named 'reports-bucket'.",
  'format': 'json'},
 {'task': "Write a regular expression that matches valid AWS EC2 instance IDs, which start with 'i-' followed by either 8 or 17 lowercase hexadecimal characters.",
  'format': 'regex'}]

## 2. Running the Eval

**Definition:** Running the eval means taking every test case in the dataset, merging it into our prompt, sending it to Claude, and collecting the outputs together with a score. We build it from three small functions, each with one job:

- **`run_prompt`** — takes one test case, builds the prompt, calls Claude, and returns the raw output.
- **`run_test_case`** — calls `run_prompt`, then *grades* the output and returns a result dictionary (`output`, `test_case`, `score`).
- **`run_eval`** — loops over the whole dataset, calling `run_test_case` for each case.

Keeping them separate means we can later swap in a better prompt (`run_prompt`) or a better grading strategy (`run_test_case`) without touching the rest.

For now, grading is a placeholder that gives every output a score of `10`. Sections 3 and 4 replace it with real graders.

In [6]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [7]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {"output": output, "test_case": test_case, "score": score}

In [8]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [9]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

[
  {
    "output": "Here's a function that counts the objects in an S3 bucket using boto3. It uses a paginator so it works correctly for buckets with more than 1,000 objects (the maximum returned by a single `list_objects_v2` call).\n\n```python\nimport boto3\n\ndef count_s3_objects(bucket_name: str) -> int:\n    \"\"\"Return the number of objects stored in the given S3 bucket.\"\"\"\n    s3 = boto3.client(\"s3\")\n    paginator = s3.get_paginator(\"list_objects_v2\")\n    count = 0\n    for page in paginator.paginate(Bucket=bucket_name):\n        count += page.get(\"KeyCount\", 0)\n    return count\n```\n\n**Usage:**\n\n```python\nif __name__ == \"__main__\":\n    print(count_s3_objects(\"my-example-bucket\"))\n```\n\n**Notes:**\n- Credentials and region are picked up from the standard boto3 configuration (environment variables, `~/.aws/config`, IAM role, etc.).\n- The caller needs the `s3:ListBucket` permission on the bucket.\n- `KeyCount` reports the number of keys in each page, so

## 3. Model-Based Grading

**Definition:** In model-based grading, we send each output to *another* Claude call whose only job is to evaluate it and return a score. It is the most flexible grader — it can judge qualities that are hard to check with code, such as correctness of reasoning, completeness, or tone.

A good grading prompt does three things:

- **Gives the grader a role and the full context** — the original task *and* the solution, wrapped in XML tags so they are clearly separated.
- **Asks for structured output** — strengths, weaknesses, reasoning, then a numeric `score`. Putting `reasoning` *before* `score` makes the model think before it commits to a number, which gives more reliable scores.
- **Uses a JSON schema, like section 1** — so the reply always matches that shape and can go straight into `json.loads`.

Below we add `grade_by_model`, tighten `run_prompt` so Claude returns only code (removing any code fence it adds), and update `run_test_case` to use the model score.

In [10]:
# Schema for the grader's reply
GRADE_SCHEMA = {
    "type": "object",
    "properties": {
        "strengths": {"type": "array", "items": {"type": "string"}},
        "weaknesses": {"type": "array", "items": {"type": "string"}},
        "reasoning": {"type": "string"},
        "score": {"type": "integer"},
    },
    "required": ["strengths", "weaknesses", "reasoning", "score"],
    "additionalProperties": False,
}


# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>


Output Format
Provide your evaluation with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A whole number between 1-10

Keep your response concise and direct.
"""

    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = chat(messages, schema=GRADE_SCHEMA)
    return json.loads(eval_text)

In [11]:
import re


def strip_code_fence(text):
    """Remove a surrounding markdown code fence, if the model added one."""
    text = text.strip()
    match = re.match(r"^```[a-zA-Z]*\n(.*?)\n?```$", text, re.DOTALL)
    return match.group(1).strip() if match else text


# Passes a test case into Claude (asks for code only, then removes any code fence)
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
* Do not wrap the answer in a markdown code block
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return strip_code_fence(output)

In [12]:
# Execute a single test case and grade it with the model grader
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [18]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 9.333333333333334


In [19]:
results

[{'output': 'def count_s3_objects(bucket_name):\n    import boto3\n    s3 = boto3.client("s3")\n    paginator = s3.get_paginator("list_objects_v2")\n    count = 0\n    for page in paginator.paginate(Bucket=bucket_name):\n        count += page.get("KeyCount", 0)\n    return count',
  'test_case': {'task': 'Write a single Python function using boto3 that takes a bucket name and returns the number of objects stored in that S3 bucket.',
   'format': 'python'},
  'score': 9.0,
  'reasoning': 'The solution correctly solves the task. The paginator over list_objects_v2 handles buckets of any size, and summing KeyCount per page gives the correct total object count. Minor issues are the per-call client creation, the in-function import, and the lack of error handling, none of which break correctness for the stated requirement.'},
 {'output': '{\n"Version": "2012-10-17",\n"Statement": [\n{\n"Sid": "ReadOnlyReportsBucket",\n"Effect": "Allow",\n"Action": [\n"s3:GetObject",\n"s3:ListBucket"\n],\n"Res

## 4. Code-Based Grading

**Definition:** Code-based grading uses ordinary Python to check an output against a rule. It is fast, free, and completely deterministic — the right tool whenever the requirement can be checked mechanically. Here, the requirement is simple: *is the output valid JSON / valid Python / a valid regex?*

We write one small validator per format. Each returns `10` if the output is valid and `0` if not, and `grade_syntax` picks the right validator using the test case's `format` field:

- **`validate_json`** — tries `json.loads`.
- **`validate_python`** — tries `ast.parse` (parses the code without running it).
- **`validate_regex`** — tries `re.compile`.

The best evals **combine** both graders: the model grader judges quality, the code grader enforces hard rules. We average the two scores in `run_test_case`.

In [ ]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        #structured output required parameter
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [14]:
# Execute a single test case and combine the model grade with the syntax grade
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

## 5. Putting It All Together

**Definition:** The final step is to run the full pipeline and reduce it to one number — the **average score** — that we can compare between prompt versions.

We update `run_eval` to print that average, then run it over the dataset. From here the workflow is the loop described at the top of this notebook:

1. Change the prompt in `run_prompt`.
2. Re-run the cell below.
3. Keep the change only if the average score goes up.

The per-test-case `reasoning` in the printed results tells you *why* a case scored low, which is where the next prompt improvement usually comes from.

In [15]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [16]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 9.333333333333334


In [17]:
print(json.dumps(results, indent=2))

[
  {
    "output": "import boto3\n\ndef count_s3_objects(bucket_name):\n    s3 = boto3.client('s3')\n    paginator = s3.get_paginator('list_objects_v2')\n    count = 0\n    for page in paginator.paginate(Bucket=bucket_name):\n        count += page.get('KeyCount', 0)\n    return count",
    "test_case": {
      "task": "Write a single Python function using boto3 that takes a bucket name and returns the number of objects stored in that S3 bucket.",
      "format": "python"
    },
    "score": 9.0,
    "reasoning": "The solution correctly counts objects by paginating through list_objects_v2 and summing KeyCount per page, which is accurate and scales to large buckets. It is concise and meets the single-function requirement. Weaknesses are limited to lack of error handling and client creation per call, which are minor for this simple task."
  },
  {
    "output": "{\"Version\": \"2012-10-17\", \"Statement\": [{\"Sid\": \"ReadOnlyReportsBucketAccess\", \"Effect\": \"Allow\", \"Action\": [\"